In [29]:
import os
import random
import librosa
import numpy as np
import sounddevice as sd
from scipy.io.wavfile import write
from sklearn.preprocessing import StandardScaler, LabelEncoder
from tensorflow.keras.models import load_model

# Import configurations
import sys
sys.path.append(os.path.abspath('..'))
from config import DATA_PATH, PROCESSED_DATA_PATH, MODEL_SAVE_PATH, SENTENCES, SAMPLE_RATE, DURATION, N_MFCC

Loading model...
Model and Scaler ready!


In [33]:

MODEL_PATH = "/Users/ayamkattel/Desktop/MISC_LATEST/Nepali-Speech-Recognition/models/nepali_asr_model.h5"
SCALER_PATH = "/Users/ayamkattel/Desktop/MISC_LATEST/Nepali-Speech-Recognition/models/scaler.joblib"
LE_PATH = "/Users/ayamkattel/Desktop/MISC_LATEST/Nepali-Speech-Recognition/models/label_encoder.joblib"


Error processing spk01_s03.wav: [Errno 2] No such file or directory: 'spk01_s03.wav'


/var/folders/nb/6rxqchpx425dyjql5p01n12w0000gn/T/ipykernel_65415/648731308.py:9: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, sample_rate = librosa.load(
/Users/ayamkattel/Desktop/MISC_LATEST/Nepali-Speech-Recognition/.venv/lib/python3.13/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


In [113]:
import os
import librosa
import numpy as np
from config import DATA_PATH, SENTENCES, SAMPLE_RATE, DURATION
from data_prep import OFFSET,FRAME_LENGTH,HOP_LENGTH,zcr,rmse,mfcc,process_audio_array

TEST_DATA_PATH="/Users/ayamkattel/Desktop/MISC_LATEST/Nepali-Speech-Recognition/test/raw"
TEST_PROCESSED_DATA_PATH="/Users/ayamkattel/Desktop/MISC_LATEST/Nepali-Speech-Recognition/test/processed"




In [114]:

def process_and_save():
    X = []
    Y = []
    
    os.makedirs(TEST_PROCESSED_DATA_PATH, exist_ok=True)
    print(f"Extracting features, categories to process: {len(SENTENCES)}")
    
    for category in SENTENCES:        
        folder_path = os.path.join(TEST_DATA_PATH, category)
        if not os.path.isdir(folder_path):
            print(f"Skipping {category}: directory not found")
            continue
            
        print(f"\nProcessing category: {category}")
        count = 0
        for file in os.listdir(folder_path):
            if file.endswith('.wav'):
                file_path = os.path.join(folder_path, file)
                try:
                    audio, sample_rate = librosa.load(file_path, sr=SAMPLE_RATE, duration=DURATION, offset=OFFSET)
                    X.append(process_audio_array(audio, sample_rate))
                    Y.append(category)
                    count += 1
                except Exception as e:
                    print(f"Error processing {file_path}: {e}")
                    
        print(f" -> Generated {count} samples for {category}")

    X = np.array(X)
    Y = np.array(Y)
    
    if len(X) > 0:
        # --- SHUFFLING LOGIC START ---
        print("Shuffling dataset...")
        indices = np.arange(X.shape[0])
        np.random.shuffle(indices)
        
        X = X[indices]
        Y = Y[indices]
        # --- SHUFFLING LOGIC END ---

        X = np.expand_dims(X, axis=2)
        print(f"Final feature shape: {X.shape}")
        
        np.save(os.path.join(TEST_PROCESSED_DATA_PATH, 'X.npy'), X)
        np.save(os.path.join(TEST_PROCESSED_DATA_PATH, 'Y.npy'), Y)
        print("Done saving shuffled X.npy and Y.npy")
    else:
        print("No data extracted.")

process_and_save()

Extracting features, categories to process: 8

Processing category: Ma_Khusi_Xu
 -> Generated 5 samples for Ma_Khusi_Xu

Processing category: Malai_Mero_Desh_Pyaro_Lagxa
 -> Generated 5 samples for Malai_Mero_Desh_Pyaro_Lagxa

Processing category: Namaste
 -> Generated 5 samples for Namaste

Processing category: Ram_Le_Vaat_Khanxa
 -> Generated 5 samples for Ram_Le_Vaat_Khanxa

Processing category: Tapaiko_Ghar_Kaha_Xa
 -> Generated 5 samples for Tapaiko_Ghar_Kaha_Xa

Processing category: TimiLai_Kasto_Chha
 -> Generated 5 samples for TimiLai_Kasto_Chha

Processing category: Uh_Mero_Mitra_Ho
 -> Generated 5 samples for Uh_Mero_Mitra_Ho

Processing category: Yo_Hamro_AI_Ko_Project_ho
 -> Generated 5 samples for Yo_Hamro_AI_Ko_Project_ho
Shuffling dataset...
Final feature shape: (40, 4004, 1)
Done saving shuffled X.npy and Y.npy


In [117]:
import numpy as np
import os
import joblib
from tensorflow.keras.models import load_model


MODEL_PATH = "/Users/ayamkattel/Desktop/MISC_LATEST/Nepali-Speech-Recognition/models/nepali_asr_model.h5"
SCALER_PATH = "/Users/ayamkattel/Desktop/MISC_LATEST/Nepali-Speech-Recognition/models/scaler.joblib"
ENCODER_PATH = "/Users/ayamkattel/Desktop/MISC_LATEST/Nepali-Speech-Recognition/models/label_encoder.joblib"

def predict_audio_sentences():
    # 2. Load the preprocessed test data
    print("Loading data and model components...")
    try:
        X_test = np.load(os.path.join(TEST_PROCESSED_DATA_PATH, 'X.npy'))
        Y_true = np.load(os.path.join(TEST_PROCESSED_DATA_PATH, 'Y.npy'))
    except FileNotFoundError:
        print("Error: Could not find X.npy or Y.npy. Ensure process_and_save() ran successfully.")
        return

    # 3. Load the Model, Scaler, and Label Encoder
    model = load_model(MODEL_PATH)
    scaler = joblib.load(SCALER_PATH)
    label_encoder = joblib.load(ENCODER_PATH)

    # 4. Process and Scale
    print("Scaling features...")
    if len(X_test.shape) == 3:
        X_test_2d = X_test.reshape(X_test.shape[0], -1)
    else:
        X_test_2d = X_test
        
    X_test_scaled = scaler.transform(X_test_2d)

    # 5. Make Predictions (Using 2D scaled data directly!)
    print("Generating predictions...")
    prediction_probs = model.predict(X_test_scaled)
    
    # Get the index of the highest probability class for each sample
    predicted_indices = np.argmax(prediction_probs, axis=1)

    # 6. Decode predictions from numbers back to sentences
    predicted_sentences = label_encoder.inverse_transform(predicted_indices)

    # 7. Display the Results
    print("\n" + "="*40)
    print("PREDICTION RESULTS")
    print("="*40)
    
    correct_predictions = 0
    total_samples = len(Y_true)

    for i in range(total_samples):
        actual = Y_true[i]
        predicted = predicted_sentences[i]
        
        # Check if prediction matches the ground truth
        is_correct = actual == predicted
        match_symbol = "✅" if is_correct else "❌"
        
        if is_correct:
            correct_predictions += 1
            
        print(f"Sample {i+1}:")
        print(f"  Actual:    {actual}")
        print(f"  Predicted: {predicted} {match_symbol}\n")

    # Calculate and print overall accuracy
    if total_samples > 0:
        accuracy = (correct_predictions / total_samples) * 100
        print(f"Overall Test Accuracy: {accuracy:.2f}% ({correct_predictions}/{total_samples} correct)")

# Run the prediction
if __name__ == "__main__":
    predict_audio_sentences()

Loading data and model components...


Scaling features...
Generating predictions...
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step

PREDICTION RESULTS
Sample 1:
  Actual:    Uh_Mero_Mitra_Ho
  Predicted: Ma_Khusi_Xu ❌

Sample 2:
  Actual:    Yo_Hamro_AI_Ko_Project_ho
  Predicted: Ma_Khusi_Xu ❌

Sample 3:
  Actual:    Yo_Hamro_AI_Ko_Project_ho
  Predicted: Ram_Le_Vaat_Khanxa ❌

Sample 4:
  Actual:    Tapaiko_Ghar_Kaha_Xa
  Predicted: Tapaiko_Ghar_Kaha_Xa ✅

Sample 5:
  Actual:    Ram_Le_Vaat_Khanxa
  Predicted: Ma_Khusi_Xu ❌

Sample 6:
  Actual:    TimiLai_Kasto_Chha
  Predicted: Yo_Hamro_AI_Ko_Project_ho ❌

Sample 7:
  Actual:    TimiLai_Kasto_Chha
  Predicted: Ram_Le_Vaat_Khanxa ❌

Sample 8:
  Actual:    Namaste
  Predicted: Ram_Le_Vaat_Khanxa ❌

Sample 9:
  Actual:    Ma_Khusi_Xu
  Predicted: Ma_Khusi_Xu ✅

Sample 10:
  Actual:    Malai_Mero_Desh_Pyaro_Lagxa
  Predicted: Ram_Le_Vaat_Khanxa ❌

Sample 11:
  Actual:    Ma_Khusi_Xu
  Predicted: Ram_Le_Vaat_Khanxa ❌

Sample 12:
  Actual:    Ma_Khusi_Xu
  Predicted: Uh_Mero_Mitra_Ho ❌


In [132]:
!pip install pyaudio

zsh:1: /Users/ayamkattel/Desktop/MISC_LATEST/Nepali-Speech-Recognition/.venv/bin/pip: bad interpreter: /Users/ayamkattel/Desktop/Nepali-Speech-Recognition/.venv/bin/python3.13: no such file or directory


In [138]:
import os
import time
import librosa
import numpy as np
import pyaudio
import joblib
from tensorflow.keras.models import load_model

# --- 1. CONFIGURATIONS ---
# Model expected settings
SAMPLE_RATE = 22050
DURATION = 4.224
FRAME_LENGTH = 2048
HOP_LENGTH = 512

# File paths
MODEL_PATH = 'nepali_model_artifacts/nepali_lstm_model.h5'
SCALER_PATH = 'nepali_model_artifacts/scaler.joblib'
ENCODER_PATH = 'nepali_model_artifacts/label_encoder.joblib'

# --- 2. LOAD ARTIFACTS ---
print("Loading Model, Scaler, and Encoder...")
try:
    model = load_model(MODEL_PATH)
    scaler = joblib.load(SCALER_PATH)
    le = joblib.load(ENCODER_PATH)
    print("✅ All artifacts loaded successfully!\n")
except Exception as e:
    print(f"❌ Error loading artifacts: {e}")
    print("Please ensure the artifacts folder is in the same directory as this script.")
    exit()

# --- 3. FEATURE EXTRACTOR ---
def process_live_audio(audio, sample_rate):
    """Must perfectly match the Kaggle sequence extraction logic."""
    audio, _ = librosa.effects.trim(audio, top_db=20)
    if len(audio) > 0:
        audio = librosa.util.normalize(audio)
        
    target_length = int(sample_rate * DURATION)
    if len(audio) < target_length:
        audio = np.pad(audio, (0, max(0, target_length - len(audio))), "constant")
    else:
        audio = audio[:target_length]
    
    zcr_val = librosa.feature.zero_crossing_rate(audio, frame_length=FRAME_LENGTH, hop_length=HOP_LENGTH).T
    rmse_val = librosa.feature.rms(y=audio, frame_length=FRAME_LENGTH, hop_length=HOP_LENGTH).T
    mfcc_val = librosa.feature.mfcc(y=audio, sr=sample_rate, n_mfcc=13, hop_length=HOP_LENGTH).T
    
    sequence = np.hstack((zcr_val, rmse_val, mfcc_val))
    return sequence

# --- 4. HARDCODED MAC RECORDING ---
def record_audio_mac(duration=DURATION, target_sr=SAMPLE_RATE):
    p = pyaudio.PyAudio()
    
    # EXACT MACBOOK AIR MIC SETTINGS
    RATE = 44100 
    CHANNELS = 1 
    DEVICE_INDEX = 1 
    
    CHUNK = 1024
    FORMAT = pyaudio.paFloat32

    print(f"🔴 RECORDING FOR {duration} SECONDS... SPEAK NOW!")
    
    # Open stream directly to your MacBook Air Mic
    stream = p.open(format=FORMAT, 
                    channels=CHANNELS, 
                    rate=RATE, 
                    input=True, 
                    input_device_index=DEVICE_INDEX,
                    frames_per_buffer=CHUNK)

    frames = []

    # Read audio chunks
    for i in range(0, int(RATE / CHUNK * duration)):
        data = stream.read(CHUNK, exception_on_overflow=False)
        frames.append(np.frombuffer(data, dtype=np.float32))

    print("✅ Recording complete. Processing...\n")

    # Stop and cleanly close the stream
    stream.stop_stream()
    stream.close()
    p.terminate()

    # Combine chunks into one continuous array
    audio_data = np.hstack(frames)

    # Downsample from Mac's 44100 Hz to the AI's 22050 Hz mathematically
    if RATE != target_sr:
        audio_data = librosa.resample(y=audio_data, orig_sr=RATE, target_sr=target_sr)

    return audio_data

# --- 5. PREDICTION PIPELINE ---
def live_predict():
    print("🎙️ Get ready to speak...")
    time.sleep(1)
    
    # 1. Record 
    audio = record_audio_mac()
    
    # 2. Extract features -> Shape: (time_steps, 15)
    features = process_live_audio(audio, SAMPLE_RATE)
    
    # 3. Scale features using the Kaggle scaler
    features_scaled = scaler.transform(features)
    
    # 4. Reshape for LSTM model -> Shape: (1, time_steps, 15)
    lstm_input = np.expand_dims(features_scaled, axis=0)
    
    # 5. Predict
    predictions = model.predict(lstm_input, verbose=0)
    predicted_index = np.argmax(predictions, axis=1)[0]
    confidence = np.max(predictions) * 100
    
    # 6. Decode label
    predicted_word = le.inverse_transform([predicted_index])[0]
    
    print("========================================")
    print("         LIVE PREDICTION RESULT         ")
    print("========================================")
    print(f"  Predicted:  {predicted_word}")
    print(f"  Confidence: {confidence:.2f}%")
    print("========================================\n")

if __name__ == "__main__":
    while True:
        live_predict()
        
        user_input = input("Press ENTER to record again, or type 'q' to quit: ")
        if user_input.lower() == 'q':
            break

Loading Model, Scaler, and Encoder...


/Users/ayamkattel/Desktop/MISC_LATEST/Nepali-Speech-Recognition/.venv/lib/python3.13/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/ayamkattel/Desktop/MISC_LATEST/Nepali-Speech-Recognition/.venv/lib/python3.13/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


✅ All artifacts loaded successfully!

🎙️ Get ready to speak...
🔴 RECORDING FOR 4.224 SECONDS... SPEAK NOW!
✅ Recording complete. Processing...

         LIVE PREDICTION RESULT         
  Predicted:  Ma_Khusi_Xu
  Confidence: 54.10%

🎙️ Get ready to speak...
🔴 RECORDING FOR 4.224 SECONDS... SPEAK NOW!
✅ Recording complete. Processing...

         LIVE PREDICTION RESULT         
  Predicted:  Ma_Khusi_Xu
  Confidence: 65.21%

🎙️ Get ready to speak...
🔴 RECORDING FOR 4.224 SECONDS... SPEAK NOW!
✅ Recording complete. Processing...

         LIVE PREDICTION RESULT         
  Predicted:  Ram_Le_Vaat_Khanxa
  Confidence: 96.80%

🎙️ Get ready to speak...
🔴 RECORDING FOR 4.224 SECONDS... SPEAK NOW!
✅ Recording complete. Processing...

         LIVE PREDICTION RESULT         
  Predicted:  Namaste
  Confidence: 98.64%

🎙️ Get ready to speak...
🔴 RECORDING FOR 4.224 SECONDS... SPEAK NOW!
✅ Recording complete. Processing...

         LIVE PREDICTION RESULT         
  Predicted:  Ma_Khusi_Xu
  Conf

In [135]:
import pyaudio

p = pyaudio.PyAudio()
print("🎙️ AVAILABLE MICROPHONES:\n" + "="*30)

for i in range(p.get_device_count()):
    dev_info = p.get_device_info_by_index(i)
    
    # Check if the device has input channels (meaning it's a microphone)
    if dev_info.get('maxInputChannels') > 0:
        print(f"🆔 Device Index: {i}")
        print(f"🎤 Name: {dev_info.get('name')}")
        print(f"🎛️ Max Channels: {int(dev_info.get('maxInputChannels'))}")
        print(f"📻 Native Rate: {int(dev_info.get('defaultSampleRate'))} Hz")
        print("-" * 30)

p.terminate()

🎙️ AVAILABLE MICROPHONES:
🆔 Device Index: 0
🎤 Name: Ultima atom 520 pro
🎛️ Max Channels: 1
📻 Native Rate: 8000 Hz
------------------------------
🆔 Device Index: 1
🎤 Name: MacBook Air Microphone
🎛️ Max Channels: 1
📻 Native Rate: 44100 Hz
------------------------------
